# Mini Project 4
Nama: Faraday Barr Fatahillah

### Tahap 1

Tahap 1 adalah memuat semua kredensial dan endpoint dari file `.env` menggunakan `python-dotenv`. Variabel yang dibutuhkan:
- **Azure OpenAI**: API key, endpoint, API version, nama model, dan nama embedding
- **Pinecone**: API key dan nama index

Setelah memuat variabel, setiap variabel akan diberikan status `OK` atau `MISSING` jadi terlihat variabel yang tidak ada.

In [ ]:
# Import library yang dibutuhkan
import os
from dotenv import load_dotenv # Untuk ENV

# Ambil data dari file environment
load_dotenv()

# Declare key dari azure yang dibutuhkan
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

AZURE_CHAT_DEPLOYMENT = os.getenv("AZURE_CHAT_DEPLOYMENT")
AZURE_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")

# Declare key dari pinecone yang dibutuhkan
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")

# Cek semua isi variabel key
for name, value in {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    "AZURE_CHAT_DEPLOYMENT": AZURE_CHAT_DEPLOYMENT,
    "AZURE_EMBEDDING_DEPLOYMENT": AZURE_EMBEDDING_DEPLOYMENT,
    "PINECONE_API_KEY": PINECONE_API_KEY
}.items():
    print(f"{name:30s} -> {'OK' if value else 'MISSING'}")

AZURE_OPENAI_API_KEY           -> OK
AZURE_OPENAI_ENDPOINT          -> OK
AZURE_OPENAI_API_VERSION       -> OK
AZURE_CHAT_DEPLOYMENT          -> OK
AZURE_EMBEDDING_DEPLOYMENT     -> OK
PINECONE_API_KEY               -> OK


### Tahap 2

Inisialisasi dua client Azure via langchain:
- `chat_llm`: Model AzureChatOpenAI dengan `temperature=0` dan maksimal token 1024 untuk semua jawaban yang generated.
- `embeddings`: Model untuk mengubah text menjadi vector yang sangat dense.

In [3]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

chat_llm = AzureChatOpenAI(
    azure_deployment=AZURE_CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
    max_completion_tokens=1024 
)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

test_llm = chat_llm.invoke(input="Who is Michael Faraday?")
print(f"Answer: {test_llm.content}")

test_embed = embeddings.embed_query("Mitsubishi")
print(f"Embedding dimension: {len(test_embed)}")

c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Answer: Michael Faraday was a 19th-century English scientist who made major discoveries in **electricity, magnetism, and chemistry**.

He is best known for:
- **Discovering electromagnetic induction** — the principle behind electric generators and transformers
- **Developing the concept of electric and magnetic fields**
- **Discovering benzene**
- **Inventing the Faraday cage**, which blocks external electric fields

Faraday was largely self-educated and became one of the most influential experimental scientists in history. His work laid much of the foundation for modern electrical engineering and physics.
Embedding dimension: 1536


In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="./data_mobil",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True
)

raw_docs = loader.load()
print(f"Total dokumen: {len(raw_docs)}")
print([f"  - {doc.metadata["source"]} ({len(doc.page_content)} chars)" for doc in raw_docs])

C:\Users\bobe\AppData\Local\Temp\ipykernel_32916\4009427078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
100%|██████████| 16/16 [00:00<00:00, 5333.30it/s]

Total dokumen: 16
['  - data_mobil\\destinator_review.txt (7587 chars)', '  - data_mobil\\destinator_spec.txt (11179 chars)', '  - data_mobil\\l100_ev_spec.txt (12308 chars)', '  - data_mobil\\l100_ev_review.txt (9073 chars)', '  - data_mobil\\l300_review.txt (8217 chars)', '  - data_mobil\\triton_review.txt (8336 chars)', '  - data_mobil\\l300_spec.txt (12461 chars)', '  - data_mobil\\pajero_sport_spec.txt (11389 chars)', '  - data_mobil\\pajero_sport_review.txt (7652 chars)', '  - data_mobil\\xforce_spec.txt (10776 chars)', '  - data_mobil\\xpander_cross_review.txt (8099 chars)', '  - data_mobil\\triton_spec.txt (12390 chars)', '  - data_mobil\\xforce_review.txt (7301 chars)', '  - data_mobil\\xpander_cross_spec.txt (11675 chars)', '  - data_mobil\\xpander_spec.txt (11909 chars)', '  - data_mobil\\xpander_review.txt (8242 chars)']


### Tahap 3

Mengiterasi semua file untuk menambahkan metadata sesuai dengan nama file:
- `doc_type`: tentukan `"spec"` atau `"review"` pada nama file.
- `model_name`: diambil dari nama file dan menghapuskan `_spec` / `_review`.
- `brand`: hardcode menjadi `"mitsubishi"`.

In [5]:
for doc in raw_docs:
    filename = os.path.basename(doc.metadata["source"]).lower()
    name_clean = filename.replace(".txt", "")

    if "spec" in name_clean:
        doc.metadata["doc_type"] = "spec"
        model_name = name_clean.replace("_spec", "").replace("spec_", "")
    elif "review" in name_clean:
        doc.metadata["doc_type"] = "review"
        model_name = name_clean.replace("_review", "").replace("review_", "")
   
    doc.metadata["model_name"] = model_name.strip("_")
    doc.metadata["brand"] = "mitsubishi"

print("Contoh metadata setelah enrichment:")
for doc in raw_docs[:3]:
    print(f"   {doc.metadata}")
    

Contoh metadata setelah enrichment:
   {'source': 'data_mobil\\destinator_review.txt', 'doc_type': 'review', 'model_name': 'destinator', 'brand': 'mitsubishi'}
   {'source': 'data_mobil\\destinator_spec.txt', 'doc_type': 'spec', 'model_name': 'destinator', 'brand': 'mitsubishi'}
   {'source': 'data_mobil\\l100_ev_spec.txt', 'doc_type': 'spec', 'model_name': 'l100_ev', 'brand': 'mitsubishi'}


### Tahap 4

Membagi dokumen menjadi chunks yang overlapping menggunakan `RecursiveCharacterTextSplitter`:
1. `chunk_size`: 500 karakter maksimum per chunk
2. `chunk_overlap`: 75 character sebagai connector antar chunk
3. `separators`: membagi sesuai paragraf, baris, titik, spasi, dan string kosong secara berurutan.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=75,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len
)

chunks = splitter.split_documents(raw_docs)
print(f"Total chunks: {len(chunks)}")
print(f"Dari {len(raw_docs)} dokumen rata-rata {len(chunks)//len(raw_docs)} chunk/dokumen")
print(f"\nSample chunk ke-1:")
print(f"Metadata : {chunks[0].metadata}")
print(f"Content  : {chunks[0].page_content[:200]}...")

Total chunks: 372
Dari 16 dokumen rata-rata 23 chunk/dokumen

Sample chunk ke-1:
Metadata : {'source': 'data_mobil\\destinator_review.txt', 'doc_type': 'review', 'model_name': 'destinator', 'brand': 'mitsubishi'}
Content  : Mitsubishi Destinator resmi diluncurkan di pasar Indonesia dengan kisaran harga yang kompetitif antara Rp 280 - 340 jutaan (on-the-road Jakarta).
Aspek yang akan Anda sukai dari mobil ini:
Desain ekst...


### Tahap 5

Koneksikan ke Pinecone menggunakan API key. Jika target index belum ada, maka di buat pada AWS `us-east-1` dengan:
- Dimension: 1 536 (sesuai dengan output Azure OpenAI `text-embedding-ada-002`).
- Metric: cosine similarity.

In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if PINECONE_INDEX not in pc.list_indexes().names():
    pc.create_index(
    name=PINECONE_INDEX, 
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

pine_index = pc.Index(PINECONE_INDEX)

### Tahap 6

Upload semua chunk yang sudah di embedding bertahap (50 data per upload) menggunakan `PineconeVectorStore.from_documents` dari batch pertama, kemudian menggunakan `vectorstore.add_documents` untuk batch selanjutnya. Diberikan sleep timer 5 untuk menghindari rate-limit errors.

In [8]:
from langchain_pinecone import PineconeVectorStore
import time

BATCH_SIZE = 50

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks[:BATCH_SIZE],
    embedding=embeddings,
    index_name=PINECONE_INDEX
)

for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"Batch {i//BATCH_SIZE + 1}: {len(batch)} chunks uploaded")
    time.sleep(5)

print(f"Total {len(chunks)} saved di Pinecone")

Batch 2: 50 chunks uploaded
Batch 3: 50 chunks uploaded
Batch 4: 50 chunks uploaded
Batch 5: 50 chunks uploaded
Batch 6: 50 chunks uploaded
Batch 7: 50 chunks uploaded
Batch 8: 22 chunks uploaded
Total 372 saved di Pinecone


Testing Query

In [9]:
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX,
    embedding=embeddings
)

test_query = "Spesifikasi Mitsubishi X-Force"
results = vectorstore.similarity_search_with_score(test_query, k=3)

for i, (doc, score) in enumerate(results):
    print(f"[{i+1}] Test Query: {test_query}")
    print(f"Model  : {doc.metadata.get('model_name', 'unknown')}")
    print(f"Type   : {doc.metadata.get('doc_type', 'unknown')}")
    print(f"Content: {doc.page_content[:120]}...")

[1] Test Query: Spesifikasi Mitsubishi X-Force
Model  : xforce
Type   : spec
Content: Rem depan seluruh varian menggunakan jenis Ventilated Disc, sedangkan rem belakang mengadopsi Solid Disc.
Ukuran dimensi...
[2] Test Query: Spesifikasi Mitsubishi X-Force
Model  : xforce
Type   : spec
Content: Rem depan seluruh varian menggunakan jenis Ventilated Disc, sedangkan rem belakang mengadopsi Solid Disc.
Ukuran dimensi...
[3] Test Query: Spesifikasi Mitsubishi X-Force
Model  : xforce
Type   : spec
Content: Semua varian Mitsubishi Xforce tidak dibekali dengan Side Stepper karena tinggi dek luar yang masuk akal.
Semua varian M...


### Tahap 7

Membuat hybrid retriever menggunakan BM25 dan Dense search menggunakan `EnsembleRetriever`

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from typing import List

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

Testing hybrid retrieval test

In [12]:
test_query = "kapasitas mesin dan torsi Xpander"
hybrid_results = hybrid_retriever.invoke(test_query)

print(f"Hybrid Search — Query: '{test_query}'")
print(f"{len(hybrid_results)} dokumen terpilih setelah RRF re-ranking\n")

for i, doc in enumerate(hybrid_results):
    print(f"[{i+1}]\nModel  : {doc.metadata.get('model_name', '?')}")
    print(f"Type   : {doc.metadata.get('doc_type', '?')}")
    print(f"Content: {doc.page_content[:150]}...\n")

Hybrid Search — Query: 'kapasitas mesin dan torsi Xpander'
15 dokumen terpilih setelah RRF re-ranking

[1]
Model  : xpander_cross
Type   : spec
Content: Torsi maksimal dari mesin mitsubishi Xpander Cross mencapai 141 Newton meters pada putaran 4000 Revolutions Per Minute (RPM).
Seluruh varian mitsubish...

[2]
Model  : xpander
Type   : spec
Content: Mesin 1.499 cc ini digunakan secara seragam baik pada varian GLS, Exceed, Sport, maupun Ultimate.
Mesin mitsubishi Xpander menghasilkan max power sebe...

[3]
Model  : xpander_cross
Type   : spec
Content: Tidak ada pilihan varian bermesin Diesel untuk seluruh jajaran produk mitsubishi Xpander Cross.
Mesin mitsubishi Xpander Cross memiliki kapasitas disp...

[4]
Model  : xpander_cross
Type   : spec
Content: Seluruh mesin mitsubishi Xpander Cross memiliki jumlah konfigurasi silinder sebanyak 4.0 silinder segaris.
Sistem penggerak semua varian mitsubishi Xp...

[5]
Model  : xpander_cross
Type   : spec
Content: Seluruh varian mitsubishi Xpander

### Tahap 8

Membuat RAG Pipeline menggunakan LangChain Expression Language:
- `RAG_PROMPT`: template prompt untuk menginstruksikan LLM menjawab sesuai konteks dan dalam Bahasa Indonesia
- `format_docs`: untuk memformat dokumen yang diambil dan diberikan label
- `rag_chain`: mengambil dan memformat konteks secara paralel dengan pertanyaan yang diteruskan, lalu memasukkan keduanya ke dalam LLM
- `rag_chain_with_sources`: memperluas `rag_chain` untuk mengembalikan dokumen mentah diambil bersama jawabannya
- `rag_with_sources`: membungkus dan mengembalikan sitasi yang berisi query, answers, dan dokumen yang diambil dalam bentuk string

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from typing import Dict

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are an AI assistant who is an expert on Mitsubishi cars.
     Answer users' questions ONLY based on the context of the provided document.
     If the information is not in the context, say, "That information is not available in my document."
     Provide clear, accurate, and easy-to-understand answers in Indonesian.
     Include technical details if relevant."""),
    ("human",
     """Context Document:
        {context}

        Question: {question}""")
])

def format_docs(docs: List) -> str:
    formatted = []
    for i, doc in enumerate(docs):
        model = doc.metadata.get('model_name', 'unknown').upper()
        dtype = doc.metadata.get('doc_type', 'general')
        formatted.append(f"[Sumber {i+1} — {model} {dtype}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# Full LCEL pipeline: retriever → format → prompt → llm → parse
rag_chain = (
    RunnableParallel(
        context  = hybrid_retriever | RunnableLambda(format_docs),
        question = RunnablePassthrough()
    )
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

rag_chain_with_sources = (
    RunnableParallel(
        answer         = rag_chain,
        retrieved_docs = hybrid_retriever,
    )
)

def rag_with_sources(query: str) -> Dict:
    result = rag_chain_with_sources.invoke(query)
    return {
        "query":          query,
        "answer":         result["answer"],
        "retrieved_docs": result["retrieved_docs"],
        "context":        format_docs(result["retrieved_docs"]),
    }

Testing

In [14]:
test_q = "Apa saja fitur utama Mitsubishi Xpander?"
result = rag_with_sources(test_q)

print(f"Pertanyaan: {result['query']}")
print(f"\nJawaban:\n{result['answer']}")
print(f"\nDokumen sumber ({len(result['retrieved_docs'])} chunk):")
for doc in result['retrieved_docs']:
    print(f"   - {doc.metadata.get('model_name')} [{doc.metadata.get('doc_type')}]")

Pertanyaan: Apa saja fitur utama Mitsubishi Xpander?

Jawaban:
Fitur utama Mitsubishi Xpander yang tercantum di dokumen adalah:

- Touchscreen sebagai pusat pengoperasian sistem hiburan kabin
- Rear Window Wiper
- Rear Window Washer
- Child Safety Locks pada kedua pintu baris kedua
- Tombol Engine Start/Stop Button, kecuali varian GLS
- Voice Commands melalui integrasi sistem multimedia ponsel pintar
- Mekanisme bukaan pintu bagasi belakang secara manual yang ringan
- Desain Dynamic Shield Grille dengan fins berkelir matte black atau chrome tergantung varian
- Indikator Tailgate Ajar Warning untuk pintu kargo bagasi
- Traction Control (TCL)
- Height Adjustable Driver Seat mekanis yang bisa diatur naik-turun secara manual
- Rear Window Defogger
- Anti-Theft Alarm
- Indikator Door Ajar Warning
- Digital Odometer pada layar panel instrumen
- Ruang penyimpanan serbaguna di bawah lantai kargo bagasi belakang
- Radio AM/FM
- Bluetooth Connectivity untuk musik nirkabel dan panggilan telepon
-

### Tahap 9

Bagian ini mengiterasi daftar yang telah ditentukan sebelumnya yang berisi lima pertanyaan terkait Mitsubishi dan memanggil fungsi `rag_with_sources` untuk setiap pertanyaan. Untuk setiap pertanyaan, jawaban akan ditampilkan bersama dengan tiga referensi sumber unik teratas (nama model dan jenis dokumen) yang berkontribusi terhadap jawaban tersebut.

In [15]:
FIVE_QUESTIONS = [
    "Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi Xpander?",
    "Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sport?",
    "Bagaimana ulasan pengguna tentang konsumsi bahan bakar Mitsubishi Xpander?",
    "Apa saja varian yang tersedia untuk Mitsubishi Pajero Sport dan perbedaannya?",
    "Berdasarkan review, apa kelebihan dan kekurangan utama Mitsubishi Xpander?",
]

qa_results = []

print("=" * 70)
print("  🚗 MITSUBISHI KNOWLEDGE ASSISTANT — 5 PERTANYAAN")
print("=" * 70)

for i, question in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n[{i}/5] {question}")
    print("-" * 60)

    result = rag_with_sources(question)
    qa_results.append(result)

    print(f"{result['answer']}")
    print(f"\nSumber: ", end="")
    sources = [f"{d.metadata.get('model_name')}[{d.metadata.get('doc_type')}]"
               for d in result['retrieved_docs'][:3]]
    print(", ".join(set(sources)))
    print("=" * 70)

  🚗 MITSUBISHI KNOWLEDGE ASSISTANT — 5 PERTANYAAN

[1/5] Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi Xpander?
------------------------------------------------------------
Mitsubishi Xpander memiliki kapasitas mesin **1499 cc** dan menghasilkan tenaga maksimum **104 BHP pada 6000 RPM**.

Sumber: xpander[spec], xpander_cross[spec]

[2/5] Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sport?
------------------------------------------------------------
Perbedaan fitur keselamatan antara **Xpander** dan **Pajero Sport** yang ada di dokumen adalah:

### Fitur keselamatan yang sama-sama dimiliki
Keduanya sama-sama memiliki:
- **Speed Sensing Auto Door Lock**
- **ISOFIX Child Seat Mounts**
- **Hill Start Assist (HSA)**  
- **Impact Sensing Auto Door Unlock**

### Fitur keselamatan yang ada di Pajero Sport
Pajero Sport memiliki beberapa fitur keselamatan tambahan yang disebutkan di dokumen, yaitu:
- **ABS** tidak disebutkan di Pajero Sport, tetapi ada **EBD**
- **Bra

### Tahap 10

Implementasi evaluasi LLM-based menggunakan 3 metrik:
1. Faithfullness: Persentase klaim jawaban yang secara langsung didukung oleh konteks yang diperoleh (anti-halusinasi)
2. Answer Relevance: Seberapa baik jawaban tersebut menjawab pertanyaan awal (kelengkapan + relevansi)
3. Context Precision: Persentase potongan data yang berhasil diambil yang benar-benar diperlukan untuk menjawab pertanyaan tersebut

In [16]:
import json
from langchain_core.output_parsers import StrOutputParser

FAITHFULNESS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an RAG Evaluator system.
Your task is to evaluate whether each claim in the ANSWER is supported by the CONTEXT.

ONLY return a VALID JSON file (without markdown):
{{"score": <0.0-1.0>, "supported_claims": <total supported claims>,
"total_claims": <total claims>, "reasoning": "<short reasoning>"}}"""),
    ("human", "CONTEXT:\n{context}\n\ANSWER:\n{answer}")
])

RELEVANCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an RAG Evaluator system.
Your task is to evaluate how relevant the ANSWER to the QUESTION.

ONLY return a VALID JSON file (without markdown):
{{"score": <0.0-1.0>, "is_complete": <true/false>,
"is_on_topic": <true/false>, "reasoning": "<short reasoning>"}}"""),
    ("human", "QUESTION:\n{query}\n\nANSWER:\n{answer}")
])

PRECISION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an RAG Evaluator system.
Your task is to check whether each retrieved chunk is ABSOLUTELY NECESSARY to answer the question.

ONLY return a VALID JSON file (without markdown):
{{"precision": <0.0-1.0>, "assessments": [{{"chunk": 1, "relevant": true, "reason": "..."}}]}}"""),
    ("human", "QUESTION:\n{query}\n\nCHUNKS:\n{chunks_text}")
])

def _parse_json(response: str) -> dict:
    clean = response.strip().replace("```json", "").replace("```", "")
    return json.loads(clean)

faithfulness_chain = FAITHFULNESS_PROMPT | chat_llm | StrOutputParser() | RunnableLambda(_parse_json)
relevance_chain    = RELEVANCE_PROMPT    | chat_llm | StrOutputParser() | RunnableLambda(_parse_json)
precision_chain    = PRECISION_PROMPT    | chat_llm | StrOutputParser() | RunnableLambda(_parse_json)

def full_ragas_evaluation(query: str) -> Dict:
    result     = rag_with_sources(query)
    chunks_text = "\n".join([f"Chunk {i+1}: {d.page_content}" for i, d in enumerate(result["retrieved_docs"])])

    try:
        faith = faithfulness_chain.invoke({"context": result["context"], "answer": result["answer"]})
    except Exception:
        faith = {"score": 0.0, "error": "parse failed"}

    try:
        rel = relevance_chain.invoke({"query": query, "answer": result["answer"]})
    except Exception:
        rel = {"score": 0.0, "error": "parse failed"}

    try:
        prec = precision_chain.invoke({"query": query, "chunks_text": chunks_text})
    except Exception:
        prec = {"precision": 0.0, "error": "parse failed"}

    f_score = faith.get("score",     0)
    r_score = rel.get("score",       0)
    p_score = prec.get("precision",  0)

    print(f"\n📝 Query   : {query}")
    print(f"💬 Answer  : {result['answer'][:200]}...")
    print(f"\n{'='*60}")
    print("📊 RAGAS EVALUATION")
    print(f"{'='*60}")
    print(f"Faithfulness      : {f_score:.2f}  — {faith.get('reasoning', '')[:80]}")
    print(f"Answer Relevance  : {r_score:.2f}  — {rel.get('reasoning', '')[:80]}")
    print(f"Context Precision : {p_score:.2f}")
    print(f"{'='*60}")

    return {
        "query":             query,
        "answer":            result["answer"],
        "retrieved_docs":    result["retrieved_docs"],
        "faithfulness":      f_score,
        "answer_relevance":  r_score,
        "context_precision": p_score,
    }

<>:11: SyntaxWarning: invalid escape sequence '\A'
<>:11: SyntaxWarning: invalid escape sequence '\A'
C:\Users\bobe\AppData\Local\Temp\ipykernel_32916\2766568948.py:11: SyntaxWarning: invalid escape sequence '\A'
  ("human", "CONTEXT:\n{context}\n\ANSWER:\n{answer}")


### Tahap 11

Menjalankan `full_ragas_evaluation` pada masing-masing dari lima pertanyaan yang telah ditentukan secara berurutan, dengan jeda 2 detik di antara setiap panggilan. Semua hasil dikumpulkan dalam `eval_results` untuk ringkasan yang ditampilkan pada sel berikutnya.

In [17]:
eval_results = []

print("Memulai evaluasi RAGAS untuk 5 pertanyaan\n")

for i, q in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n[{i}/5] Evaluasi: {q[:60]}...")
    eval_result = full_ragas_evaluation(q)
    eval_results.append(eval_result)
    time.sleep(2)

Memulai evaluasi RAGAS untuk 5 pertanyaan


[1/5] Evaluasi: Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi...

📝 Query   : Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi Xpander?
💬 Answer  : Mitsubishi Xpander memiliki kapasitas mesin **1499 cc** dan menghasilkan tenaga maksimum **104 BHP pada 6000 RPM**....

📊 RAGAS EVALUATION
Faithfulness      : 1.00  — Kedua klaim didukung langsung oleh Sumber 10 (kapasitas mesin 1499 cc) dan Sumbe
Answer Relevance  : 1.00  — Jawaban langsung menjawab pertanyaan tentang kapasitas mesin dan tenaga Mitsubis
Context Precision : 1.00

[2/5] Evaluasi: Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sp...

📝 Query   : Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sport?
💬 Answer  : Perbedaan fitur keselamatan antara **Xpander** dan **Pajero Sport** yang ada di dokumen adalah sebagai berikut:

### Fitur keselamatan yang sama-sama dimiliki
Keduanya sama-sama memiliki:
- **Speed Se...

📊 RAGAS EVALUATION
Fait

### Tahap 12

Menampilkan tabel yang telah diformat yang memperlihatkan skor Keakuratan, Relevansi Jawaban, dan Ketepatan Konteks untuk masing-masing dari lima pertanyaan yang dievaluasi, diikuti dengan rata-rata per kolom. Keterangan di bagian bawah menjelaskan arti dari setiap metrik (0,0 = terburuk, 1,0 = terbaik).

In [18]:
print("\n" + "=" * 90)
print(f"{'RAGAS EVALUATION SUMMARY — MITSUBISHI RAG SYSTEM':^90}")
print("=" * 90)
print(f"  {'No':<4} {'Pertanyaan':<45} {'Faith':>7} {'Relev':>7} {'Prec':>7}")
print("-" * 90)

avg_faith = avg_rel = avg_prec = 0

for i, r in enumerate(eval_results, 1):
    q_short = r['query'][:43] + ".." if len(r['query']) > 43 else r['query']
    f = r['faithfulness']
    rv = r['answer_relevance']
    p = r['context_precision']
    avg_faith += f
    avg_rel   += rv
    avg_prec  += p
    print(f"  {i:<4} {q_short:<45} {f:>7.2f} {rv:>7.2f} {p:>7.2f}")

n = len(eval_results)
print("-" * 90)
print(f"  {'RATA-RATA':<49} {avg_faith/n:>7.2f} {avg_rel/n:>7.2f} {avg_prec/n:>7.2f}")
print("=" * 90)
print()
print("📌 Interpretasi skor (0.0 = buruk, 1.0 = sempurna):")
print("   Faithfulness      : Apakah jawaban didukung oleh konteks (anti-halusinasi)")
print("   Answer Relevance  : Apakah jawaban menjawab pertanyaan dengan tepat")
print("   Context Precision : Apakah dokumen yang di-retrieve memang relevan")


                     RAGAS EVALUATION SUMMARY — MITSUBISHI RAG SYSTEM                     
  No   Pertanyaan                                      Faith   Relev    Prec
------------------------------------------------------------------------------------------
  1    Berapa kapasitas mesin dan tenaga yang diha..    1.00    1.00    1.00
  2    Apa perbedaan fitur keselamatan antara Xpan..    0.95    0.93    0.80
  3    Bagaimana ulasan pengguna tentang konsumsi ..    1.00    0.86    0.25
  4    Apa saja varian yang tersedia untuk Mitsubi..    0.96    0.93    0.18
  5    Berdasarkan review, apa kelebihan dan kekur..    1.00    0.97    0.75
------------------------------------------------------------------------------------------
  RATA-RATA                                            0.98    0.94    0.60

📌 Interpretasi skor (0.0 = buruk, 1.0 = sempurna):
   Faithfulness      : Apakah jawaban didukung oleh konteks (anti-halusinasi)
   Answer Relevance  : Apakah jawaban menjawab pertanyaan 

### Tahap 13

Menjalankan seluruh alur kerja RAG pada pertanyaan yang ditentukan pengguna (`YOUR_QUESTION`) untuk membandingkan model Xpander dan Xpander Cross. Menampilkan pertanyaan, jawaban yang dihasilkan, serta daftar bernomor dari potongan sumber (nama model, jenis dokumen, dan 80 karakter pertama dari konten) yang digunakan untuk menyusun jawaban tersebut.

In [19]:
YOUR_QUESTION = "Bagaimana perbandingan harga dan fitur antara Xpander dan Xpander Cross?"

result = rag_with_sources(YOUR_QUESTION)

print(f"❓ Pertanyaan  : {result['query']}")
print(f"\n💡 Jawaban:\n{result['answer']}")
print(f"\n📚 Sumber dokumen yang digunakan:")
for i, doc in enumerate(result['retrieved_docs'], 1):
    m = doc.metadata
    print(f"   [{i}] {m.get('model_name','?').upper()} — {m.get('doc_type','?')} "
          f"| '{doc.page_content[:80]}...'")

❓ Pertanyaan  : Bagaimana perbandingan harga dan fitur antara Xpander dan Xpander Cross?

💡 Jawaban:
Berdasarkan dokumen:

### Harga
- **Xpander**: berkisar **Rp 260–315 jutaan**.
- **Xpander Cross**: berkisar **Rp 320–350 jutaan**.

Jadi, **Xpander Cross lebih mahal** dibanding Xpander.

### Perbandingan fitur
**Fitur yang sama-sama ada:**
- **Keyless Operation System (KOS)**  
- **Tachometer digital**
- **ABS**
- **Central Locking otomatis**
- **Driver Airbag + Passenger Airbag**
- **Electric Power Steering (EPS)** pada Xpander Cross, sedangkan untuk Xpander fitur ini tidak disebutkan dalam dokumen
- **Bluetooth Connectivity**
- **Android Auto via kabel**
- **Apple CarPlay**
- **Radio AM/FM**

**Fitur yang disebut khusus pada Xpander Cross:**
- **AC Digital**
- **Heater**
- **9 inci Touchscreen**
- **Rear Window Wiper**
- **Rear Window Washer**
- **Traction Control (TCL)**
- **Height Adjustable Driver Seat**
- **Height Adjustable Front Seat Belts**
- **T-Shape LED Headlamps**
- **Aut